# Item Effectiveness Volcano Plot

For every upgrade item, tests whether holding it is associated with a higher (or lower) win rate
specifically when facing a chosen hero on the enemy team.

- **x-axis:** log₂(Odds Ratio) — positive means the item is associated with winning against the target hero
- **y-axis:** −log₁₀(BH-adjusted p-value) — higher is more statistically significant
- Shape encodes item slot: ○ weapon  □ vitality  △ spirit
- Only high-rank matches (avg_badge ≥ 100 on both teams)

## Configuration

Set `HERO_NAME` to any hero in the dataset. Run the cell below first to see all available hero names.

In [ ]:
# ── Set the target hero here ──────────────────────────────────────────────────
HERO_NAME = "Vyper"
# ─────────────────────────────────────────────────────────────────────────────

HIGH_RANK     = 100   # minimum avg_badge for both teams
MIN_N_WITH    = 20    # minimum teams holding item while facing the hero
MIN_N_WITHOUT = 5     # minimum teams not holding it
THRESH_P      = 0.05  # BH-adjusted p-value significance threshold
THRESH_OR     = 1.2   # odds-ratio threshold for 'small effect' classification

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

pd.set_option('display.max_columns', None)

OTHER_TEAM = {'Team0': 'Team1', 'Team1': 'Team0'}

In [ ]:
items_lookup  = pd.read_parquet('data/public_items.parquet')
heroes_lookup = pd.read_parquet('data/public_heroes.parquet')

for col in items_lookup.columns:
    items_lookup[col] = items_lookup[col].apply(lambda x: x.decode() if isinstance(x, bytes) else x)

upgrade_item_ids = set(items_lookup[items_lookup['type'] == 'upgrade']['id'])
item_id_to_name  = items_lookup.set_index('id')['name'].to_dict()
item_slot        = items_lookup[items_lookup['type'] == 'upgrade'].set_index('name')['slot_type'].to_dict()

hero_id_to_name  = heroes_lookup.set_index('id')['name'].to_dict()
hero_name_to_id  = {v: k for k, v in hero_id_to_name.items()}

print('Available heroes:')
print(sorted(hero_name_to_id.keys()))

In [ ]:
if HERO_NAME not in hero_name_to_id:
    raise ValueError(f"Hero '{HERO_NAME}' not found. Check spelling against the list above.")

TARGET_HERO_ID = np.int64(hero_name_to_id[HERO_NAME])
print(f"Target hero: {HERO_NAME} (id={TARGET_HERO_ID})")

## Load match data

In [ ]:
DATA_FILES = ['data/collected_matches.jsonl', 'data/collected_matches_gap.jsonl']

player_rows, item_rows = [], []

for path in DATA_FILES:
    if not os.path.exists(path):
        print(f'  {path} not found — skipping')
        continue
    print(f'  Loading {path} ...')
    with open(path) as f:
        for line in f:
            try:
                match = json.loads(line)
            except json.JSONDecodeError:
                continue
            if match['average_badge_team0'] < HIGH_RANK or match['average_badge_team1'] < HIGH_RANK:
                continue
            mid = match['match_id']
            won = match['winning_team']
            for p in match['players']:
                team    = p['team']
                hero_id = np.int64(p['hero_id'])
                player_rows.append({'match_id': mid, 'team': team,
                                    'hero_id': hero_id, 'won': int(team == won)})
                for item in p['items']:
                    iid = np.uint32(item['item_id'])
                    if iid not in upgrade_item_ids:
                        continue
                    item_rows.append({'match_id': mid, 'team': team,
                                      'won': int(team == won), 'item_id': iid,
                                      'held': item['sold_time_s'] == 0})

players_df = pd.DataFrame(player_rows)
items_df   = pd.DataFrame(item_rows)
items_df['item_name'] = items_df['item_id'].map(item_id_to_name)

n_matches = players_df['match_id'].nunique()
print(f'High-rank matches loaded: {n_matches:,}')
print(f'Player rows: {len(players_df):,}  |  Item rows: {len(items_df):,}')

## Build per-team feature table

In [ ]:
# Flag teams that face the target hero
hero_on_enemy = (
    players_df[players_df['hero_id'] == TARGET_HERO_ID][['match_id', 'team']]
    .assign(enemy_target=1)
    .assign(team=lambda d: d['team'].map(OTHER_TEAM))
)

outcomes = players_df[['match_id', 'team', 'won']].drop_duplicates()

# Wide table: one row per (match, team), columns = item held (0/1)
held_wide = (
    items_df[items_df['held']]
    .groupby(['match_id', 'team', 'item_name'])
    .size().gt(0).astype(int)
    .rename('held')
    .reset_index()
    .pivot_table(index=['match_id', 'team'], columns='item_name',
                 values='held', aggfunc='max', fill_value=0)
    .reset_index()
)
held_wide.columns.name = None

team_df = outcomes.merge(hero_on_enemy, on=['match_id', 'team'], how='left')
team_df['enemy_target'] = team_df['enemy_target'].fillna(0).astype(int)

n_facing = team_df['enemy_target'].sum()
print(f'Teams facing {HERO_NAME}: {n_facing:,}  /  {len(team_df):,} total team-slots')

## Chi-square tests — win rate by item, among teams facing the target hero

In [ ]:
volcano_df_raw = team_df.merge(held_wide, on=['match_id', 'team'], how='left')
item_cols = [c for c in held_wide.columns if c not in ('match_id', 'team')]
volcano_df_raw[item_cols] = volcano_df_raw[item_cols].fillna(0).astype(int)

facing = volcano_df_raw[volcano_df_raw['enemy_target'] == 1]

rows = []
for item in item_cols:
    with_item    = facing[facing[item] == 1]['won']
    without_item = facing[facing[item] == 0]['won']
    n1, n0 = len(with_item), len(without_item)
    if n1 < MIN_N_WITH or n0 < MIN_N_WITHOUT:
        continue

    a = int(with_item.sum())
    b = n1 - a
    c = int(without_item.sum())
    d = n0 - c
    if b == 0 or c == 0:
        continue

    chi2, p, _, _ = chi2_contingency(np.array([[a, b], [c, d]]), correction=False)
    or_val = (a * d) / (b * c)
    rows.append({
        'item':       item,
        'slot_type':  item_slot.get(item, 'unknown'),
        'n_with':     n1,
        'n_without':  n0,
        'wr_with':    round(float(with_item.mean()), 3),
        'wr_without': round(float(without_item.mean()), 3),
        'wr_delta':   round(float(with_item.mean() - without_item.mean()), 3),
        'or':         or_val,
        'log2_or':    np.log2(or_val),
        'p_raw':      p,
    })

vdf = pd.DataFrame(rows)
print(f'Items with enough observations: {len(vdf)}')

In [ ]:
_, vdf['p_bh'],   _, _ = multipletests(vdf['p_raw'], method='fdr_bh')
_, vdf['p_bonf'], _, _ = multipletests(vdf['p_raw'], method='bonferroni')
vdf['neg_log10_bh'] = -np.log10(vdf['p_bh'].clip(lower=1e-300))

log2_thresh = np.log2(THRESH_OR)

def categorise(row):
    if row['p_bh'] >= THRESH_P:
        return 'non-significant'
    if row['log2_or'] >  log2_thresh:
        return 'beneficial'
    if row['log2_or'] < -log2_thresh:
        return 'detrimental'
    return 'significant-small-effect'

vdf['category'] = vdf.apply(categorise, axis=1)
print('Category counts:')
print(vdf['category'].value_counts().to_string())

## Volcano plot

In [ ]:
palette = {
    'beneficial':               'steelblue',
    'detrimental':              'tomato',
    'significant-small-effect': 'orange',
    'non-significant':          '#cccccc',
}
z_order      = {'non-significant': 1, 'significant-small-effect': 2, 'detrimental': 3, 'beneficial': 4}
slot_markers = {'weapon': 'o', 'vitality': 's', 'spirit': '^', 'unknown': 'D'}

fig, ax = plt.subplots(figsize=(13, 9))

for cat, grp in vdf.groupby('category'):
    for slot, sgrp in grp.groupby('slot_type'):
        ax.scatter(sgrp['log2_or'], sgrp['neg_log10_bh'],
                   c=palette[cat],
                   marker=slot_markers.get(slot, 'o'),
                   alpha=0.65, s=35,
                   zorder=z_order.get(cat, 2),
                   label=f'{cat} / {slot}' if cat != 'non-significant' else None)

ax.axhline(-np.log10(THRESH_P), color='dimgrey', linestyle='--', linewidth=1,
           label=f'p_BH = {THRESH_P}')
ax.axvline( log2_thresh, color='dimgrey', linestyle=':', linewidth=0.8)
ax.axvline(-log2_thresh, color='dimgrey', linestyle=':', linewidth=0.8)
ax.axvline(0,            color='black',   linewidth=0.5)

sig = vdf[vdf['category'].isin(['beneficial', 'detrimental', 'significant-small-effect'])]
for _, row in sig.iterrows():
    ax.annotate(
        row['item'],
        xy=(row['log2_or'], row['neg_log10_bh']),
        xytext=(6, 4), textcoords='offset points',
        fontsize=7, color='black',
        arrowprops=dict(arrowstyle='-', color='grey', lw=0.5),
    )

ax.set_xlabel(f'log₂(Odds Ratio)  —  positive: item associated with winning vs {HERO_NAME}', fontsize=10)
ax.set_ylabel('−log₁₀(p-value, BH corrected)', fontsize=10)
ax.set_title(
    f'{HERO_NAME} — Item Win Rate Associations (volcano plot)\n'
    f'({len(vdf)} items tested; {n_facing:,} team-slots facing {HERO_NAME}; BH FDR; OR threshold {THRESH_OR}x)\n'
    f'shape = slot type  (○ weapon  □ vitality  △ spirit)',
    fontsize=11,
)

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8, framealpha=0.85)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f'volcano_{HERO_NAME.lower().replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
plt.show()

## Significant items summary table

In [ ]:
sig_items = (
    vdf[vdf['category'] != 'non-significant']
    .sort_values('log2_or', ascending=False)
    [['item', 'slot_type', 'n_with', 'wr_with', 'wr_without', 'wr_delta', 'or', 'p_bh', 'category']]
    .rename(columns={'wr_delta': 'ΔWR', 'or': 'OR', 'p_bh': 'p (BH)', 'n_with': 'n held'})
    .reset_index(drop=True)
)
display(sig_items.round(3))